# Regional Sales Comparison

## Project Overview
This notebook compares sales performance across geographic regions using the Sample Superstore dataset. We analyze revenue, order count, average order value, category mix, and growth trends to identify strong and weak regions and the likely drivers behind their differences.

## Learning Objectives
- Compare multi-dimensional performance across regions using consistent metrics.
- Decompose regional revenue into volume, pricing, and mix effects.
- Build year-over-year and month-over-month growth comparisons by region.
- Identify actionable drivers behind regional differences.

## Problem Statement
A retail business operates across multiple regions. Leadership needs to know which regions are performing well, which are lagging, and *why* — so they can allocate resources, set targets, and investigate root causes.

## Why This Project Matters
Regional analysis is one of the most common requests in business analytics. Understanding geographic performance differences helps with territory planning, marketing budget allocation, staffing decisions, and inventory distribution.

## Dataset Overview
We use the repo-local `Sample - Superstore.xls` workbook (~10K retail orders). Key fields: `Region`, `State`, `Category`, `Sub-Category`, `Sales`, `Profit`, `Quantity`, `Discount`, `Order Date`, `Segment`, `Order ID`, `Customer ID`.

## Dataset Source and License Notes
Widely circulated Superstore training dataset already in this repository. No PII. Confirm redistribution terms for external use.

## Environment Setup

In [ ]:
import importlib, subprocess, sys

for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'xlrd', 'scipy']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('Environment ready.')

## Imports

In [ ]:
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
np.random.seed(42)

## Configuration / Constants

In [ ]:
def locate_workspace_root(start_path):
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'README.md').exists():
            return candidate
    return start_path

WORKSPACE_ROOT = locate_workspace_root(Path.cwd())
DATASET_PATH = WORKSPACE_ROOT / 'Time Series Analysis' / 'Time Series Forecasting' / 'Sample - Superstore.xls'

print(f'Workspace root: {WORKSPACE_ROOT}')
print(f'Dataset path  : {DATASET_PATH}')

## Dataset Loading

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}')

raw_df = pd.read_excel(DATASET_PATH)
print(f'Raw shape: {raw_df.shape}')
raw_df.head()

## Data Validation Checks

In [ ]:
expected = {'Order ID', 'Order Date', 'Region', 'Category', 'Sub-Category',
            'Sales', 'Profit', 'Quantity', 'Discount', 'Segment', 'State', 'Customer ID'}
missing = expected - set(raw_df.columns)
if missing:
    raise ValueError(f'Missing columns: {missing}')

print(f'Rows: {len(raw_df)}')
print(f'Nulls:\n{raw_df[list(expected)].isna().sum().to_string()}')
print(f'\nDuplicate rows: {raw_df.duplicated().sum()}')
print(f'Regions: {sorted(raw_df["Region"].unique())}')

## Data Cleaning and Preparation

In [ ]:
df = raw_df.drop_duplicates().copy()
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
df = df.dropna(subset=['Order Date', 'Sales', 'Region']).copy()

df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['YearMonth'] = df['Order Date'].dt.to_period('M')
df['Profit Margin %'] = np.where(df['Sales'] != 0, df['Profit'] / df['Sales'] * 100, np.nan)

print(f'Clean rows: {len(df)}')
print(f'Date range: {df["Order Date"].min().date()} – {df["Order Date"].max().date()}')

## Exploratory Data Analysis

### Regional Summary — Revenue, Orders, and Profit
The core comparison table: total revenue, profit, order count, average order value, and profit margin by region.

In [ ]:
region_summary = df.groupby('Region').agg(
    Revenue=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order ID', 'nunique'),
    Customers=('Customer ID', 'nunique'),
    Quantity=('Quantity', 'sum'),
    Avg_Discount=('Discount', 'mean'),
).sort_values('Revenue', ascending=False)

region_summary['AOV'] = region_summary['Revenue'] / region_summary['Orders']
region_summary['Margin %'] = np.where(region_summary['Revenue'] != 0,
                                       region_summary['Profit'] / region_summary['Revenue'] * 100, np.nan)
region_summary['Revenue Share %'] = region_summary['Revenue'] / region_summary['Revenue'].sum() * 100

print('Regional Summary:')
print(region_summary.round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
plot_df = region_summary.reset_index()

axes[0].bar(plot_df['Region'], plot_df['Revenue'] / 1e3, color='steelblue', edgecolor='black')
axes[0].set_title('Total Revenue ($K)')
axes[0].set_ylabel('Revenue ($K)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

axes[1].bar(plot_df['Region'], plot_df['Orders'], color='teal', edgecolor='black')
axes[1].set_title('Order Count')

axes[2].bar(plot_df['Region'], plot_df['AOV'], color='darkorange', edgecolor='black')
axes[2].set_title('Average Order Value ($)')

axes[3].bar(plot_df['Region'], plot_df['Margin %'], color='seagreen', edgecolor='black')
axes[3].set_title('Profit Margin %')

for ax in axes:
    ax.set_xlabel('Region')
plt.tight_layout()
plt.show()

### Category Mix by Region
Does the product mix vary across regions? Category concentration can explain revenue and margin differences.

In [ ]:
cat_rev = df.pivot_table(values='Sales', index='Region', columns='Category', aggfunc='sum', fill_value=0)
cat_share = cat_rev.div(cat_rev.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

cat_rev.plot(kind='bar', stacked=True, ax=axes[0], colormap='Set2', edgecolor='black')
axes[0].set_title('Revenue by Category per Region')
axes[0].set_ylabel('Revenue ($)')
axes[0].legend(title='Category')

cat_share.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2', edgecolor='black')
axes[1].set_title('Category Share (%) per Region')
axes[1].set_ylabel('Share %')
axes[1].legend(title='Category')

plt.tight_layout()
plt.show()

print('Category share (%) by region:')
print(cat_share.round(1).to_string())

### Sub-Category Performance by Region
Drill deeper to find which sub-categories drive or drag each region.

In [ ]:
subcat_region = df.groupby(['Region', 'Sub-Category']).agg(
    Revenue=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
).reset_index()
subcat_region['Margin %'] = np.where(subcat_region['Revenue'] != 0,
                                       subcat_region['Profit'] / subcat_region['Revenue'] * 100, np.nan)

# Top 5 sub-categories by revenue per region
for reg in sorted(df['Region'].unique()):
    top5 = subcat_region[subcat_region['Region'] == reg].nlargest(5, 'Revenue')
    print(f'\n{reg} — Top 5 Sub-Categories:')
    print(top5[['Sub-Category', 'Revenue', 'Profit', 'Margin %']].to_string(index=False))

### Segment Mix by Region
Does the customer segment composition (Consumer, Corporate, Home Office) differ by region?

In [ ]:
seg_rev = df.pivot_table(values='Sales', index='Region', columns='Segment', aggfunc='sum', fill_value=0)
seg_share = seg_rev.div(seg_rev.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
seg_share.plot(kind='bar', stacked=True, ax=ax, colormap='viridis', edgecolor='black')
ax.set_title('Segment Share (%) by Region')
ax.set_ylabel('Share %')
ax.legend(title='Segment')
plt.tight_layout()
plt.show()

print(seg_share.round(1).to_string())

## Growth Trend Analysis

### Monthly Revenue Trend by Region
Overlay regional revenue trends to spot diverging or converging trajectories.

In [ ]:
monthly_region = df.groupby(['YearMonth', 'Region'], as_index=False).agg(Revenue=('Sales', 'sum'))
monthly_region['date'] = monthly_region['YearMonth'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(16, 5))
for reg in sorted(df['Region'].unique()):
    subset = monthly_region[monthly_region['Region'] == reg]
    ax.plot(subset['date'], subset['Revenue'], marker='.', label=reg, linewidth=1.5)
ax.set_title('Monthly Revenue by Region')
ax.set_ylabel('Revenue ($)')
ax.legend(title='Region')
plt.tight_layout()
plt.show()

### Year-over-Year Growth by Region
Compare annual growth rates to identify which regions are gaining or losing momentum.

In [ ]:
annual_region = df.groupby(['Year', 'Region'], as_index=False).agg(
    Revenue=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order ID', 'nunique'),
)

yoy_pivot = annual_region.pivot(index='Year', columns='Region', values='Revenue')
yoy_growth = yoy_pivot.pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

yoy_pivot.plot(kind='bar', ax=axes[0], width=0.7)
axes[0].set_title('Annual Revenue by Region')
axes[0].set_ylabel('Revenue ($)')

yoy_growth.plot(kind='bar', ax=axes[1], width=0.7, colormap='coolwarm')
axes[1].set_title('Year-over-Year Revenue Growth (%)')
axes[1].set_ylabel('Growth %')
axes[1].axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

print('YoY Growth (%) by Region:')
print(yoy_growth.round(1).to_string())

### Seasonality Comparison
Do regions have different seasonal patterns? Compare average monthly revenue by region.

In [ ]:
seasonal = df.groupby(['Month', 'Region']).agg(Revenue=('Sales', 'sum')).reset_index()
years_count = df['Year'].nunique()
seasonal['Avg Monthly Revenue'] = seasonal['Revenue'] / years_count

seasonal_pivot = seasonal.pivot(index='Month', columns='Region', values='Avg Monthly Revenue')

fig, ax = plt.subplots(figsize=(14, 5))
seasonal_pivot.plot(ax=ax, marker='o')
ax.set_title('Average Monthly Revenue by Region (Seasonality)')
ax.set_ylabel('Avg Revenue ($)')
ax.set_xlabel('Month')
ax.set_xticks(range(1, 13))
plt.tight_layout()
plt.show()

## Discount Behavior by Region
Do some regions rely more heavily on discounts? This can explain margin differences.

In [ ]:
disc_region = df.groupby('Region').agg(
    Avg_Discount=('Discount', 'mean'),
    Pct_Discounted=('Discount', lambda x: (x > 0).mean() * 100),
    Avg_Discount_When_Given=('Discount', lambda x: x[x > 0].mean() if (x > 0).any() else 0),
).round(3)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(disc_region.index, disc_region['Avg_Discount'] * 100, color='coral', edgecolor='black')
axes[0].set_title('Average Discount (%) by Region')
axes[0].set_ylabel('Discount %')

axes[1].bar(disc_region.index, disc_region['Pct_Discounted'], color='mediumpurple', edgecolor='black')
axes[1].set_title('% of Orders Discounted by Region')
axes[1].set_ylabel('% Orders')

plt.tight_layout()
plt.show()

print(disc_region.to_string())

## State-Level Drill-Down
Surface the top and bottom states within each region by revenue and margin.

In [ ]:
state_perf = df.groupby(['Region', 'State']).agg(
    Revenue=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order ID', 'nunique'),
).reset_index()
state_perf['Margin %'] = np.where(state_perf['Revenue'] != 0,
                                    state_perf['Profit'] / state_perf['Revenue'] * 100, np.nan)

for reg in sorted(df['Region'].unique()):
    subset = state_perf[state_perf['Region'] == reg].sort_values('Revenue', ascending=False)
    print(f'\n=== {reg} — Top 5 States ===')
    print(subset.head(5)[['State', 'Revenue', 'Profit', 'Margin %', 'Orders']].to_string(index=False))
    bottom = subset.tail(3)
    if len(bottom) > 0:
        print(f'  Bottom 3: {", ".join(bottom["State"].tolist())}')

## Composite Regional Ranking
Rank regions on multiple normalized metrics to get a fair overall score.

In [ ]:
rank_cols = ['Revenue', 'Margin %', 'Orders', 'AOV', 'Customers']
rank_df = region_summary[rank_cols].copy()

# Normalize each metric 0-1
for col in rank_cols:
    mn, mx = rank_df[col].min(), rank_df[col].max()
    rank_df[f'{col}_norm'] = (rank_df[col] - mn) / (mx - mn) if mx != mn else 0.5

norm_cols = [c for c in rank_df.columns if c.endswith('_norm')]
rank_df['Composite Score'] = rank_df[norm_cols].mean(axis=1)
rank_df = rank_df.sort_values('Composite Score', ascending=False)

print('Composite Regional Ranking:')
print(rank_df[['Revenue', 'Margin %', 'Orders', 'Composite Score']].round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(rank_df.index, rank_df['Composite Score'], color='steelblue', edgecolor='black')
ax.set_xlabel('Composite Score (0-1)')
ax.set_title('Regional Composite Ranking')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Executive Summary Insights

In [ ]:
top_reg = region_summary.index[0]
bottom_reg = region_summary.index[-1]
overall_aov = df['Sales'].sum() / df['Order ID'].nunique()

insights = [
    f'Strongest region by revenue: {top_reg} (${region_summary.loc[top_reg, "Revenue"]:,.0f}, '
    f'{region_summary.loc[top_reg, "Revenue Share %"]:.1f}% share)',
    f'Weakest region by revenue: {bottom_reg} (${region_summary.loc[bottom_reg, "Revenue"]:,.0f})',
    f'Highest margin region: {region_summary["Margin %"].idxmax()} '
    f'({region_summary["Margin %"].max():.1f}%)',
    f'Lowest margin region: {region_summary["Margin %"].idxmin()} '
    f'({region_summary["Margin %"].min():.1f}%)',
    f'Overall average order value: ${overall_aov:,.2f}',
    f'Number of regions: {len(region_summary)}',
]
for item in insights:
    print(f'  * {item}')

## Limitations
- Analysis is limited to a single sample dataset; regional patterns may not generalize.
- No external context (population, economic conditions, marketing spend per region) to explain differences.
- State-level analysis is limited by sample sizes in smaller states.
- Profit margins are derived from the dataset and may not reflect actual cost structures.

## Recommendations / Next Steps
- Investigate the lowest-margin region to determine if discounting, product mix, or operational costs are the primary drag.
- Correlate regional performance with marketing spend and staffing levels (external data needed).
- Set up automated regional dashboards that refresh monthly for ongoing tracking.
- Test targeted promotions in underperforming regions to measure elasticity.

## Common Mistakes
- Comparing raw revenue without normalizing for region size, population, or store count.
- Drawing causal conclusions from correlation (e.g., "discounts cause lower margins" when the causal direction may be reversed).
- Ignoring category mix effects — a region may look weak simply because it sells lower-priced categories.
- Judging growth rates without checking absolute scale — a small region can show high growth rates from a low base.

## Mini Challenge
1. Build a per-capita revenue metric using US Census state population data.
2. Find the single sub-category that drives the biggest revenue gap between the top and bottom region.
3. Compare weekend vs. weekday order patterns by region.
4. Create a revenue concentration index (HHI) by region to measure how diversified each region's category mix is.

## Final Summary / Key Takeaways
This notebook compares regional performance on revenue, profitability, order volume, category mix, growth, seasonality, and discount behavior. The composite ranking provides a balanced multi-metric view. The key discipline: always decompose aggregate differences into their component drivers (volume, price, mix, discount) before drawing conclusions.